# [Super AI Engineer Season 6] Mini Hackathon 3 Level 2
## FahMai Telephone Directory v2 -- Agentic RAG + /nothink Optimisation

**Super AI Engineer Season 6 - Level 2 Hackathon**
- Dataset: FahMai Telephone Directory (1,995 employees, 19 columns) & 300 Questions
- Model: **typhoon-v2.5-30b-a3b-instruct** (MoE, /nothink mode)
- Approach: Agentic ReAct Loop + Dual Search Tools + Code-to-Dept Lookup

---
### Why v1 scored only 51.9% -- and how v2 fixes it

| Bucket (Worst in v1) | v1 | Root Cause | v2 Fix |
|---|---|---|---|
| name_lookup | 0% | Searched "RETVP" literally; CSV has "RET","VP" separately | Code-decoder + dual-field search |
| vp_identity | 11% | Same code issue | Pre-built DEPT_CODE map |
| evp_identity_by_description | 25% | Too many irrelevant refusals | Cleaner prompt, broader search |
| casual_name_lookup | 22% | Nickname search didn't strip whitespace | regex + case-insensitive fix |
| subsidiary_md | 0% | Unknown -- needs investigation | search_by_field tool |

### Key Techniques
1. **/nothink** -- Prepend `/nothink` in system prompt to disable chain-of-thought (Qwen3 MoE trick = 2-3x faster, cheaper, more deterministic)
2. **Code Decoder** -- Maps "RETVP" -> Department="RET", Position level="VP" before calling grep_csv
3. **Dual Tools** -- `grep_csv` (broad substring) + `search_by_field` (exact column filter)
4. **Tight Refusals** -- Only refuse truly forbidden topics; never refuse valid directory queries

### Notebook Outline
1. Setup & Imports
2. Data Loading & EDA
3. Tool Definitions (grep_csv + search_by_field + decode helper)
4. Generic Agent Loop (ReAct)
5. System Prompt v2 (/nothink + refined rules)
6. Inference & Submission
7. Local Validation (grade.py)

# 1. Setup & Imports
### 1.1 Install & configure Typhoon v2.5 (MoE, /nothink mode)

In [1]:
!pip install -q openai pandas tqdm

import os, json, re, time, warnings
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from openai import OpenAI

warnings.filterwarnings('ignore')

# --- API Key (Kaggle / Colab / Local) ----------------------------------------
from kaggle_secrets import UserSecretsClient
TYPHOON_API_KEY = UserSecretsClient().get_secret("TYPHOON_API_KEY")

# --- Model config ------------------------------------------------------------
# typhoon-v2.5-30b-a3b-instruct = Qwen3-based MoE
# /nothink in system prompt disables hidden chain-of-thought => faster + cheaper
MODEL      = 'typhoon-v2.5-30b-a3b-instruct'
MAX_TOKENS = 2048   # short answers only; no need for 8192

client = OpenAI(api_key=TYPHOON_API_KEY, base_url='https://api.opentyphoon.ai/v1')

# --- Verify connection -------------------------------------------------------
try:
    pong = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'system', 'content': '/nothink'},
                  {'role': 'user',   'content': 'Reply with the single word: Ready'}],
        max_tokens=10,
    )
    print("Connection OK:", pong.choices[0].message.content)
except Exception as e:
    print("Connection FAILED:", e)

Connection OK: Ready


# 2. Data Loading & EDA
### 2.1 Load employees.csv + questions.csv

In [3]:
# --- Paths (Kaggle competition vs local) ------------------------------------
KAGGLE_DIR = Path('/kaggle/input/competitions/super-ai-engineer-season-6-fahmai-telephone-directory/')
LOCAL_DIR  = Path('super-ai-engineer-season-6-fahmai-telephone-directory')
DATA_DIR   = KAGGLE_DIR if KAGGLE_DIR.exists() else LOCAL_DIR

df_emp = pd.read_csv(DATA_DIR / 'employees.csv').fillna('')
df_questions = pd.read_csv(DATA_DIR / 'questions.csv')

# Convert Phone Extension to int-string to remove ".0"
df_emp['Phone Extension'] = df_emp['Phone Extension'].apply(
    lambda x: str(int(float(x))) if x != '' else '')

print(f"Employees : {len(df_emp):,} rows x {df_emp.shape[1]} cols")
print(f"Questions : {len(df_questions):,}")
print(f"Columns   : {list(df_emp.columns)}")

Employees : 1,995 rows x 19 cols
Questions : 300
Columns   : ['Employee ID', 'Department', 'Section', 'Unit', 'Position in Thai', 'Position in English', 'First Name Thai', 'Last Name Thai', 'First Name English', 'Last Name English', 'Nickname Thai', 'Nickname English', 'Email Address', 'Phone Extension', 'Mobile No.', 'Office Location', 'Branch', 'Start Year', 'Position Level']


### 2.2 EDA -- Key distributions

In [4]:
display(df_emp.head(3))
print("--- Position Level ---"); print(df_emp['Position Level'].value_counts())
print("--- Top Departments ---"); print(df_emp['Department'].value_counts().head(8))
has_nick = (df_emp['Nickname Thai'] != '').sum()
print(f"--- Nickname coverage: {has_nick}/{len(df_emp)} ({has_nick/len(df_emp)*100:.0f}%) ---")
print("--- Question language ---"); print(df_questions['language'].value_counts())

,Employee ID,Department,Section,Unit,Position in Thai,Position in English,First Name Thai,Last Name Thai,First Name English,Last Name English,Nickname Thai,Nickname English,Email Address,Phone Extension,Mobile No.,Office Location,Branch,Start Year,Position Level
0,2699,CEO,CEO-OFF,CEO,ประธานเจ้าหน้าที่บริหาร,CHIEF EXECUTIVE OFFICER,วชิร,จิรบุญ,VACHIR,CHIRABUN,เบอร์รี่,BERRY,VACHIR.CH@FAHMAI.CO.TH,73048,,FahMai Tower 12F,BKK-R9,2016,C-level
1,8902591,CEO,CEO-OFF,CEO-EA,เลขานุการของ CEO,EXECUTIVE ASSISTANT TO CEO,อรญา,วัชรกาญจน์,ORRAYA,WATCHARAKAN,เป้,PE,ORRAYA.WA@FAHMAI.CO.TH,75665,,FahMai Tower 7F,BKK-R9,2023,Manager
2,6662,CEO,CEO-OFF,CEO-CoS,หัวหน้าสำนักงานประธาน,CHIEF OF STAFF,กิตติคุณ,พงจงรัก,KITTIKHUN,PHONGCHONGRAK,บูม,BOOM,KITTIKHUN.PH@FAHMAI.CO.TH,79367,062-174-6941,FahMai Tower 16F,BKK-R9,2017,VP


--- Position Level ---
Position Level
IC          1573
Lead         188
Manager      140
Director      63
VP            24
C-level        7
Name: count, dtype: int64
--- Top Departments ---
Department
RET    380
TEC    231
SUP    184
LOG    169
SF     129
DN     118
OPS    112
MKT    105
Name: count, dtype: int64
--- Nickname coverage: 997/1995 (50%) ---
--- Question language ---
language
th    234
en     66
Name: count, dtype: int64


# 3. Tool Definitions
### 3.1 Code Decoder -- the KEY fix for name_lookup & vp_identity (0% in v1)
Questions like "RETVP ใคร" contain a compound code: Dept abbreviation + Level.
We decode it before searching so the agent gets the right search terms.

In [5]:
# --- Department code decoder ------------------------------------------------
# Maps short dept code -> full Department column value in employees.csv
DEPT_MAP = {
    'CEO': 'CEO', 'FIN': 'FIN', 'MKT': 'MKT', 'TEC': 'TEC', 'OPS': 'OPS',
    'HR':  'HR',  'LEG': 'LEG', 'RET': 'RET', 'LOG': 'LOG', 'SUP': 'SUP',
    'SF':  'SF',  'DN':  'DN',  'KS':  'KS',  'JC':  'JC',  'GR':  'GR',
    'FIN': 'FIN', 'PRD': 'PRD', 'IT':  'IT',
}

LEVEL_MAP = {
    'EVP': 'EVP', 'VP': 'VP', 'DIR': 'Director', 'FL': 'Lead',
    'MGR': 'Manager', 'UPC': 'UPC', 'MD': 'MD',
    'CTO': 'CTO', 'CFO': 'CFO', 'COO': 'COO', 'CHRO': 'CHRO',
    'CEO': 'CEO',
}

TITLE_MAP = {
    'CEO': 'CHIEF EXECUTIVE OFFICER',
    'CFO': 'CHIEF FINANCIAL OFFICER',
    'COO': 'CHIEF OPERATING OFFICER',
    'CTO': 'CHIEF TECHNOLOGY OFFICER',
    'CHRO': 'CHIEF HUMAN RESOURCES OFFICER',
    'CSO': 'CHIEF SALES OFFICER',
    'CMO': 'CHIEF MARKETING OFFICER',
}

def decode_position_code(code_str):
    '''Split compound code like RETVP -> dept=RET, level=VP'''
    s = code_str.upper().strip()
    if s in TITLE_MAP:
        return {'title': TITLE_MAP[s], 'dept': None, 'level': s}
    for dept_len in [3, 2]:
        prefix = s[:dept_len]
        if prefix in DEPT_MAP:
            suffix = s[dept_len:]
            for lv_code, lv_name in LEVEL_MAP.items():
                if suffix == lv_code:
                    return {'dept': DEPT_MAP[prefix], 'level': lv_name, 'dept_code': prefix}
    return {}

# Smoke test
tests = ['RETVP', 'LOGFL', 'TECEVP', 'CEO', 'CHRO', 'OPSVP', 'DNVP', 'SFVP']
for t in tests:
    print(t, '->', decode_position_code(t))

RETVP -> {'dept': 'RET', 'level': 'VP', 'dept_code': 'RET'}
LOGFL -> {'dept': 'LOG', 'level': 'Lead', 'dept_code': 'LOG'}
TECEVP -> {'dept': 'TEC', 'level': 'EVP', 'dept_code': 'TEC'}
CEO -> {'title': 'CHIEF EXECUTIVE OFFICER', 'dept': None, 'level': 'CEO'}
CHRO -> {'title': 'CHIEF HUMAN RESOURCES OFFICER', 'dept': None, 'level': 'CHRO'}
OPSVP -> {'dept': 'OPS', 'level': 'VP', 'dept_code': 'OPS'}
DNVP -> {'dept': 'DN', 'level': 'VP', 'dept_code': 'DN'}
SFVP -> {'dept': 'SF', 'level': 'VP', 'dept_code': 'SF'}


### 3.2 Tool 1: `grep_csv` -- broad substring search across all columns

In [6]:
def grep_csv(pattern: str, max_results: int = 20) -> str:
    """Case-insensitive substring match across ALL columns of employees.csv."""
    p = (pattern or '').strip()
    if not p:
        return json.dumps({'error': 'empty pattern'})

    p_esc = re.escape(p)
    mask = df_emp.astype(str).apply(
        lambda col: col.str.contains(p_esc, case=False, na=False)
    ).any(axis=1)
    hits = df_emp[mask]
    total = len(hits)

    if total == 0:
        return json.dumps({'total': 0, 'note': f'No match for "{pattern}"'})

    return json.dumps({
        'total': total,
        'returned': min(total, max_results),
        'truncated': total > max_results,
        'csv': hits.head(max_results).to_csv(index=False)
    }, ensure_ascii=False)

GREP_CSV_TOOL = {
    'type': 'function',
    'function': {
        'name': 'grep_csv',
        'description': (
            f'Search FahMai employee directory ({len(df_emp)} rows). '
            'Case-insensitive substring match across ALL 19 columns: '
            'Employee ID, Department, Section, Unit, Position in Thai/English, '
            'First/Last Name Thai/English, Nickname Thai/English, Email Address, '
            'Phone Extension, Mobile No., Office Location, Branch, Start Year, Position Level. '
            'Use this for name, nickname, department, extension, mobile, email lookups. '
            'Returns JSON with total count and matching rows as CSV.'
        ),
        'parameters': {
            'type': 'object',
            'properties': {
                'pattern': {
                    'type': 'string',
                    'description': 'Keyword to search. Examples: "วชิร", "BERRY", "RET", "VP", "73048", "062-174"'
                },
                'max_results': {'type': 'integer', 'default': 20}
            },
            'required': ['pattern'],
        },
    },
}

### 3.3 Tool 2: `search_by_field` -- precise column filter (fixes vp_identity bucket)


In [7]:
def search_by_field(
    department: str = '',
    position_level: str = '',
    unit: str = '',
    section: str = '',
    position_en_contains: str = '',
    nickname_th: str = '',
    nickname_en: str = '',
    first_name_th: str = '',
    last_name_th: str = '',
    first_name_en: str = '',
    last_name_en: str = '',
    branch: str = '',
    max_results: int = 20,
) -> str:
    """Exact-field filter tool. More precise than grep_csv for known-field queries."""
    df = df_emp.copy()

    def filt(col, val):
        if val:
            return df[col].str.contains(re.escape(val.strip()), case=False, na=False)
        return pd.Series([True] * len(df), index=df.index)

    mask = (
        filt('Department', department) &
        filt('Position Level', position_level) &
        filt('Unit', unit) &
        filt('Section', section) &
        filt('Position in English', position_en_contains) &
        filt('Nickname Thai', nickname_th) &
        filt('Nickname English', nickname_en) &
        filt('First Name Thai', first_name_th) &
        filt('Last Name Thai', last_name_th) &
        filt('First Name English', first_name_en) &
        filt('Last Name English', last_name_en) &
        filt('Branch', branch)
    )
    hits = df[mask]
    total = len(hits)
    if total == 0:
        return json.dumps({'total': 0, 'note': 'No match with given filters'})
    return json.dumps({
        'total': total,
        'returned': min(total, max_results),
        'csv': hits.head(max_results).to_csv(index=False)
    }, ensure_ascii=False)

SEARCH_BY_FIELD_TOOL = {
    'type': 'function',
    'function': {
        'name': 'search_by_field',
        'description': (
            'Filter employees by specific columns. More precise than grep_csv. '
            'Use when you know which field to filter on. '
            'All parameters are optional substring matches (case-insensitive). '
            'KEY USE: department + position_level to find "RETVP" = department="RET" + position_level="VP". '
            'position_level values: C-level, VP, Director, Manager, Lead, IC. '
            'Also use position_en_contains for C-suite e.g. "CHIEF EXECUTIVE" to find CEO.'
        ),
        'parameters': {
            'type': 'object',
            'properties': {
                'department':          {'type': 'string', 'description': 'e.g. "RET", "TEC", "CEO", "FIN"'},
                'position_level':      {'type': 'string', 'description': 'e.g. "VP", "Director", "C-level", "Manager"'},
                'unit':                {'type': 'string', 'description': 'Unit code e.g. "CEO", "CFO"'},
                'section':             {'type': 'string'},
                'position_en_contains':{'type': 'string', 'description': 'Substring in English position e.g. "CHIEF OPERATING"'},
                'nickname_th':         {'type': 'string'},
                'nickname_en':         {'type': 'string'},
                'first_name_th':       {'type': 'string'},
                'last_name_th':        {'type': 'string'},
                'first_name_en':       {'type': 'string'},
                'last_name_en':        {'type': 'string'},
                'branch':              {'type': 'string'},
                'max_results':         {'type': 'integer', 'default': 20},
            },
        },
    },
}

TOOLS = [GREP_CSV_TOOL, SEARCH_BY_FIELD_TOOL]
TOOL_MAP = {
    'grep_csv': grep_csv,
    'search_by_field': search_by_field,
}

# Smoke test
r = json.loads(search_by_field(department='RET', position_level='VP'))
print("search_by_field(dept=RET, level=VP) total:", r['total'])
if r['total'] > 0:
    print(r['csv'][:300])

search_by_field(dept=RET, level=VP) total: 3
Employee ID,Department,Section,Unit,Position in Thai,Position in English,First Name Thai,Last Name Thai,First Name English,Last Name English,Nickname Thai,Nickname English,Email Address,Phone Extension,Mobile No.,Office Location,Branch,Start Year,Position Level
8000,RET,RET-HQ,RETVP,รองประธานฝ่ายเคร


# 4. Generic Agent Loop (ReAct Pattern)
### 4.1 Loop until no tool_calls or max_iters -- with retry on API errors

In [8]:
def run_agent(messages, tools, tool_map, max_iters=5, verbose=False):
    """
    ReAct loop: call model -> if tool_calls, execute them -> append results -> repeat.
    Returns final text answer when model stops calling tools.
    Reference: Agentic AI lesson Section 5, run_agent() pattern.
    """
    for i in range(max_iters):
        # --- Call LLM ---------------------------------------------------------
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=tools,
                max_tokens=MAX_TOKENS,
                temperature=0.0,
            )
        except Exception as e:
            time.sleep(3)
            try:
                resp = client.chat.completions.create(
                    model=MODEL, messages=messages, tools=tools,
                    max_tokens=MAX_TOKENS, temperature=0.0,
                )
            except Exception as e2:
                raise RuntimeError(f"API failed twice: {e2}")

        msg = resp.choices[0].message
        messages.append(msg)

        # --- No more tool calls => final answer -------------------------------
        if not msg.tool_calls:
            return (msg.content or '').strip()

        if verbose:
            for c in msg.tool_calls:
                args_preview = c.function.arguments[:60].replace('\n', ' ')
                print(f"  [iter {i}] -> {c.function.name}({args_preview})")

        # --- Execute each tool call ------------------------------------------
        for call in msg.tool_calls:
            fn = call.function.name
            try:
                args = json.loads(call.function.arguments)
            except Exception:
                args = {}
            result = str(tool_map[fn](**args)) if fn in tool_map else f"Unknown tool: {fn}"
            messages.append({'role': 'tool', 'tool_call_id': call.id, 'content': result})

    return '[max_iters reached]'

print("Agent Loop ready")

Agent Loop ready


# 5. System Prompt v2
### 5.1 Key changes from v1
- `/nothink` at the very start -- disables Qwen3 chain-of-thought (faster & more deterministic)
- Explicit instruction to use `search_by_field` for position codes like RETVP/LOGVP
- Tighter refusal rules -- only refuse truly forbidden data, never refuse valid lookups

In [9]:
SYSTEM_PROMPT_V2 = (
    '/nothink\\n\\n'
    'You are the FahMai Company Telephone Directory Assistant.\\n'
    'ALWAYS use the provided tools (grep_csv or search_by_field) to look up data.\\n'
    'NEVER guess, invent, or rely on training memory for employee details.\\n\\n'
    '== TOOL USAGE GUIDE ==\\n'
    '1. For position codes like RETVP, LOGFL, TECEVP:\\n'
    '   - Split into dept code (first 2-3 letters) and level (rest).\\n'
    '   - RET + VP  => call search_by_field(department="RET", position_level="VP")\\n'
    '   - LOG + FL  => call search_by_field(department="LOG", position_level="Lead")\\n'
    '   - TEC + EVP => call search_by_field(department="TEC", position_level="EVP")\\n'
    '2. For C-suite (CEO/CFO/COO/CTO/CHRO): call search_by_field(unit="CEO") '
    'or grep_csv("CHIEF EXECUTIVE")\\n'
    '3. For names/nicknames: use grep_csv with the name or nickname string.\\n'
    '4. For extensions/emails/mobiles: use grep_csv with the number or email.\\n'
    '5. If first search returns 0 results, try alternative keywords.\\n\\n'
    '== LANGUAGE RULE ==\\n'
    'Thai question => Thai answer. English question => English answer. No mixing.\\n\\n'
    '== REFUSAL RULES (use EXACT phrase, nothing else) ==\\n'
    'Refuse ONLY if asking for: salary/bonus/age/religion/performance review/'
    'promotion decision/disciplinary records/personal opinions/non-FahMai data.\\n'
    '  - Thai refusal: "\u0e44\u0e21\u0e48\u0e2a\u0e32\u0e21\u0e32\u0e23\u0e16\u0e43\u0e2b\u0e49\u0e02\u0e49\u0e2d\u0e21\u0e39\u0e25\u0e19\u0e35\u0e49\u0e44\u0e14\u0e49"\\n'
    '  - Thai (other company): "\u0e44\u0e21\u0e48\u0e43\u0e0a\u0e48\u0e02\u0e49\u0e2d\u0e21\u0e39\u0e25\u0e02\u0e2d\u0e07\u0e1f\u0e49\u0e32\u0e43\u0e2b\u0e21\u0e48"\\n'
    '  - Thai (opinion): "\u0e44\u0e21\u0e48\u0e2a\u0e32\u0e21\u0e32\u0e23\u0e16\u0e43\u0e2b\u0e49\u0e04\u0e27\u0e32\u0e21\u0e40\u0e2b\u0e47\u0e19\u0e44\u0e14\u0e49"\\n'
    '  - Thai (no nickname): "\u0e44\u0e21\u0e48\u0e21\u0e35\u0e0a\u0e37\u0e48\u0e2d\u0e40\u0e25\u0e48\u0e19\u0e43\u0e19\u0e23\u0e30\u0e1a\u0e1a"\\n'
    '  - Thai (not found): "\u0e44\u0e21\u0e48\u0e1e\u0e1a\u0e02\u0e49\u0e2d\u0e21\u0e39\u0e25"\\n'
    '  - English refusal: "cannot provide this information"\\n'
    '  - English (other company): "not Fahmai data"\\n'
    '  - English (opinion): "cannot offer an opinion"\\n'
    '  - English (no nickname): "no nickname on record"\\n'
    '  - English (not found): "no record found"\\n\\n'
    '== LEAKAGE RULE ==\\n'
    'When refusing, output ONLY the refusal phrase. '
    'Do NOT include Employee ID, Phone Extension, or email.\\n\\n'
    '== ANSWER STYLE ==\\n'
    'Be concise. Include both Thai and English names when available.\\n'
    'For phone queries: give the extension number.\\n'
    'For count queries: state the exact number from tool output.\\n'
)

print('System Prompt v2 ready')
print(f'Length: {len(SYSTEM_PROMPT_V2)} chars')
print('Contains /nothink:', SYSTEM_PROMPT_V2.startswith('/nothink'))

System Prompt v2 ready
Length: 2020 chars
Contains /nothink: True


# 6. Inference & Submission Generation
### 6.1 Process all 300 questions with the improved agent

In [10]:
def process_question(q_id, question, language, verbose=False):
    """Run one question through the ReAct agent loop."""
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT_V2},
        {'role': 'user',   'content': question},
    ]
    fallback = 'ไม่พบข้อมูล' if language == 'th' else 'no record found'
    try:
        ans = run_agent(messages, TOOLS, TOOL_MAP, max_iters=5, verbose=verbose)
        return ans if ans and ans not in ('[max_iters reached]', '') else fallback
    except Exception as e:
        print(f"  ERROR {q_id}: {e}")
        return fallback

# --- Run inference -----------------------------------------------------------
results = []
print(f"Running inference on {len(df_questions)} questions...")
print(f"Model: {MODEL}")
print(f"Tools: {[t['function']['name'] for t in TOOLS]}")
print()

for _, row in tqdm(df_questions.iterrows(), total=len(df_questions), desc='Answering'):
    q_id     = row['id']
    question = row['question']
    language = row['language'] if 'language' in row.index else 'th'
    answer   = process_question(q_id, question, language, verbose=False)
    results.append({'id': q_id, 'response': answer})

submission_df = pd.DataFrame(results)
submission_df.to_csv('submission_FTD.csv', index=False)
print(f"Saved {len(submission_df)} rows -> submission_FTD.csv")
display(submission_df.head(10))

Running inference on 300 questions...
Model: typhoon-v2.5-30b-a3b-instruct
Tools: ['grep_csv', 'search_by_field']



Answering:   0%|          | 0/300 [00:00<?, ?it/s]

Saved 300 rows -> submission_FTD.csv


,id,response
0,g001,The RETVP roles are:\n\n1. **Wiriya Chanchai**...
1,g002,คึกฤทธิ์ บุษราคัมวงศ์ (KUKRIT BUSARAKHAMWONG) ...
2,g004,"ไพโรจน์ มหากุล (Phairoj Mahakun), รองประธานฝ่า..."
3,g005,พงษ์กานต์ ราชชากัญญ์ (PONGKAN RAJCHAKAN) เป็น ...
4,g007,ตำแหน่ง LOGFL (หัวหน้าทีมพนักงานขับรถ) มีดังนี...
5,g008,เรืองศักดิ์ เทพเกียรติกำจร (Ruangsak Thepkiatk...
6,g009,"- ดาริกา อวุทธ์ดี (DARIKA AWUTDI), รองประธานฝ่..."
7,g011,"ณฐามน อภิชัยดี (NATHAMON APHICHAIDEE), CHRO"
8,g012,"- คะวัง กอบสุขรัตน์ (KWANG KOBSOOKRAT), รองประ..."
9,g014,"ณัฐกานต์ ศรีอารมณ์ดี (NATTHAKAN SRIAROMDEE), ต..."


# 7. Local Validation
### 7.1 Run grade.py against train_labels.json (158 public items)

In [11]:
import subprocess

grade_script  = DATA_DIR / 'grade.py'
train_labels  = DATA_DIR / 'train_labels.json'

if grade_script.exists() and train_labels.exists():
    res = subprocess.run(
        ['python', str(grade_script), 'submission_FTD.csv', str(train_labels)],
        capture_output=True, text=True, encoding='utf-8'
    )
    print("=" * 60)
    print("LOCAL GRADE RESULTS")
    print("=" * 60)
    print(res.stdout)
    if res.stderr:
        print("STDERR:", res.stderr[:500])
else:
    print("grade.py or train_labels.json not found in", DATA_DIR)

LOCAL GRADE RESULTS
Scored 158 items against /kaggle/input/competitions/super-ai-engineer-season-6-fahmai-telephone-directory/train_labels.json
Passed: 97/158 = 61.4%

Bucket                             pass/total    rate
--------------------------------------------------------
nickname_grid                    9/      17   52.9%
refuse                           8/      15   53.3%
evp_secretary                    1/       9   11.1%
vp_identity                      3/       9   33.3%
casual_name_lookup               6/       9   66.7%
evp_identity_by_code             7/       8   87.5%
evp_identity_by_description      4/       8   50.0%
name_lookup                      8/       8  100.0%
dept_listing_medium              8/       8  100.0%
dept_member_count                3/       7   42.9%
dept_listing_small               3/       6   50.0%
extension_reverse                4/       5   80.0%
hard_nickname_variant            3/       5   60.0%
evp_vs_vp_disambig               3/       4  

### 7.2 Quick response analysis

In [12]:
submission_df['len'] = submission_df['response'].str.len()
print("Response length stats:")
print(submission_df['len'].describe())
print()

# Show worst/best by length
print("Shortest (might be wrong refusals):")
display(submission_df.nsmallest(5, 'len')[['id','response']])

print("Sample responses:")
for _, row in submission_df.sample(8, random_state=7).iterrows():
    print(f"  [{row['id']}] {str(row['response'])[:90]}...")

Response length stats:
count     300.0000
mean      258.0000
std       486.4688
min        10.0000
25%        24.0000
50%        83.5000
75%       165.0000
max      2641.0000
Name: len, dtype: float64

Shortest (might be wrong refusals):


,id,response
261,g348,Admin User
11,g018,ไม่พบข้อมูล
18,g026,ไม่พบข้อมูล
19,g028,ไม่พบข้อมูล
37,g053,ไม่พบข้อมูล


Sample responses:
  [g338] ไม่สามารถให้ข้อมูลนี้ได้...
  [g082] VP วงโคจร (RETVP) คือ วิริยะ จันทชัย (WIRIYA CHANCHAI)  
ตำแหน่ง: รองประธานฝ่ายเครือข่ายร้...
  [g201] SF-EXEC มีสมาชิก 2 คน:

1. **จิรภัทร วัชรใจงาม** (Jirapat Watcharajaaingam)  
   - ตำแหน่ง...
  [g092] - วิริยะ จันทชัย (Wiriya Chanchai), ติ๊ก (TIK), โทร. 79141  
- พรไพร มหาสินธุ์ (Phonphai M...
  [g159] มีทั้งหมด 15 คนที่มีชื่อเล่นว่า "อ้อม" ดังนี้:

1. **ดอกรัก ใจงามฟ้า** (DOKRAK JAINGAMFA) ...
  [g372] ไม่พบข้อมูล...
  [g272] - HRVP: 72146  
- LEGVP: 79653  
- FINVP: 77907...
  [g333] Apologies, but I cannot provide this information....
